# Alzheimer's Disease Classification with Cross-Attention Fusion
## Multi-Task Learning with Uncertainty Estimation and Explainability
### Optimized for Kaggle T4 GPU (16GB VRAM)

A classification pipeline for Alzheimer's Disease diagnosis using multimodal brain imaging.

**Key Features:**
- **Dual 3D ResNet-18 Backbone**: Separate feature extractors for MRI and PET volumes
- **Asymmetric Cross-Attention Fusion**: Transformer-based multimodal feature fusion
- **Multi-Task Learning**: 3-class AD classification (CN/MCI/AD) + MMSE score regression
- **MC Dropout**: Monte Carlo Dropout for epistemic uncertainty quantification
- **Temperature Scaling**: Post-hoc calibration with Expected Calibration Error (ECE)
- **3D Grad-CAM**: Gradient-weighted Class Activation Mapping for explainability
- **Integrated Gradients**: Per-voxel attribution with axiomatic guarantees (Sundararajan et al., 2017)
- **Occlusion Sensitivity**: Perturbation-based region importance analysis (Zeiler & Fergus, 2014)
- **Cross-Attention Maps**: Inter-modality attention visualization
- **Class-Weighted Loss**: Handles imbalanced class distribution

**Data:** NACC preprocessed MRI-PET pairs (160 x 192 x 160, normalized to [-1, 1])

**Labels:** NACCUDSD column from 500_patients.csv (1=CN, 3=MCI, 4=AD)

In [ ]:
# ============================================================
# SETUP: Install Dependencies + Import Libraries
# ============================================================
import os, sys, gc, math, copy, random, time, json, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc,
    mean_absolute_error, mean_squared_error,
)
from scipy.ndimage import zoom as scipy_zoom
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore', category=UserWarning)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} | VRAM: {gpu.total_mem / 1e9:.1f} GB')


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

class Config:
    # ---- Paths ----
    DATA_ROOT   = '/kaggle/input/nacc-preprocessed'   # Patient volume directories
    CSV_PATH    = '/kaggle/input/nacc-preprocessed/500_patients.csv'
    SAVE_DIR    = '/kaggle/working/classification_checkpoints'

    # ---- Synthetic PET ----
    # Path to synthetic PET volumes (from diffusion model).
    # If a patient has no real PET but has synthetic, it will be used.
    # Set to '' to disable synthetic PET entirely.
    SYNTHETIC_PET_ROOT  = ''   # e.g. '/kaggle/input/synthetic-pet'
    SYNTHETIC_PET_FILE  = 'pet_synthetic.npy'  # filename inside patient dir
    REAL_PET_FILE       = 'pet_processed.npy'

    # Modality confidence: how much to trust PET relative to MRI.
    # 1.0 = full trust (real PET), lower = less trust (synthetic PET).
    # MRI always gets weight 1.0.  PET weight is:
    #   real PET    -> REAL_PET_CONFIDENCE    (default 1.0)
    #   synthetic   -> SYNTH_PET_CONFIDENCE   (default 0.3)
    REAL_PET_CONFIDENCE  = 1.0
    SYNTH_PET_CONFIDENCE = 0.3   # ~30% PET, ~70% MRI for synthetic

    # ---- Data ----
    ORIG_SHAPE     = (160, 192, 160)   # Original volume shape
    CROP_SHAPE     = (96, 112, 96)     # Center crop for GPU memory
    NUM_CLASSES    = 3                  # CN=0, MCI=1, AD=2
    LABEL_MAP      = {1: 0, 3: 1, 4: 2}  # NACCUDSD -> class index
    CLASS_NAMES    = ['CN', 'MCI', 'AD']
    MMSE_MISSING   = [88, -4, 95, 96, 97, 98]  # NACC missing codes
    VAL_SPLIT      = 0.15
    TEST_SPLIT     = 0.15

    # ---- 3D ResNet Backbone ----
    BACKBONE_CHANNELS = [64, 128, 256, 512]
    FEATURE_DIM       = 512

    # ---- Cross-Attention Fusion ----
    FUSION_HEADS    = 8
    FUSION_LAYERS   = 2
    FUSION_DROPOUT  = 0.1

    # ---- Multi-Task ----
    CLS_WEIGHT      = 1.0     # Classification loss weight
    REG_WEIGHT      = 0.3     # MMSE regression loss weight

    # ---- Training ----
    BATCH_SIZE      = 2
    EPOCHS          = 80
    LR              = 1e-4
    WEIGHT_DECAY    = 1e-4
    GRAD_CLIP       = 1.0
    WARMUP_EPOCHS   = 5
    DROPOUT         = 0.3      # Dropout rate (used for MC Dropout)

    # ---- MC Dropout (Uncertainty) ----
    MC_SAMPLES      = 30       # Number of forward passes
    MC_DROPOUT_RATE = 0.5      # Dropout rate during MC inference

    # ---- Temperature Scaling ----
    TEMP_LR         = 0.01
    TEMP_ITERS      = 50

    # ---- Evaluation ----
    VAL_EVERY       = 5
    SAVE_EVERY      = 5

    # ---- Augmentation ----
    AUG_FLIP_P      = 0.5
    AUG_ROT_DEG     = 5.0
    AUG_ROT_P       = 0.3
    AUG_NOISE_STD   = 0.02
    AUG_NOISE_P     = 0.2

    # ---- Resume ----
    RESUME_PATH     = ''

cfg = Config()
os.makedirs(cfg.SAVE_DIR, exist_ok=True)

print('=' * 60)
print('CONFIGURATION')
print('=' * 60)
for attr in sorted(vars(Config)):
    if not attr.startswith('_'):
        print(f'  {attr:22s} = {getattr(cfg, attr)}')
print('=' * 60)


## 1. Data Pipeline

### Label Mapping
| NACCUDSD | Diagnosis | Class |
|---|---|---|
| 1 | Normal Cognition (CN) | 0 |
| 2 | Impaired-not-MCI | **Excluded** |
| 3 | Mild Cognitive Impairment (MCI) | 1 |
| 4 | Dementia / Alzheimer's Disease (AD) | 2 |

### PET Source Handling (Real vs Synthetic)
The dataset automatically detects whether PET is **real** or **synthetic** (from diffusion model):
- **Real PET** (`pet_processed.npy`): full confidence = 1.0
- **Synthetic PET** (`pet_synthetic.npy`): reduced confidence = 0.3

This confidence value is passed through the entire pipeline:
1. **Fusion**: PET features are scaled by confidence before cross-attention, and the fusion gate is biased toward MRI (e.g., ~70% MRI / ~30% PET for synthetic)
2. **Training**: the model learns to rely more on MRI when PET confidence is low
3. **Inference**: predictions for synthetic-PET patients reflect appropriate uncertainty

### Preprocessing
- Volumes: 160x192x160 center-cropped to 96x112x96
- MRI and PET as separate single-channel 3D inputs
- MMSE scores normalized to [0, 1] for regression (MMSE/30)
- Class-weighted sampling to handle imbalanced data


In [ ]:
# ============================================================
# DATA LOADING + DATASET CLASS
# ============================================================

def load_patient_labels(csv_path, label_map):
    """Load CSV and map NACCUDSD to class labels."""
    df = pd.read_csv(csv_path)
    print(f'CSV loaded: {len(df)} rows, {len(df.columns)} columns')
    print(f'NACCUDSD distribution:\n{df["NACCUDSD"].value_counts().sort_index()}')

    # Filter to valid diagnoses only
    df = df[df['NACCUDSD'].isin(label_map.keys())].copy()
    df['label'] = df['NACCUDSD'].map(label_map)
    print(f'\nAfter filtering (exclude NACCUDSD=2): {len(df)} patients')
    print(f'Class distribution:')
    for udsd, cls_idx in sorted(label_map.items()):
        n = (df['label'] == cls_idx).sum()
        print(f'  NACCUDSD={udsd} -> class {cls_idx}: {n} patients')

    # MMSE: handle missing codes
    df['mmse_valid'] = ~df['NACCMMSE'].isin(cfg.MMSE_MISSING)
    df['mmse_norm'] = df['NACCMMSE'].clip(0, 30) / 30.0
    df.loc[~df['mmse_valid'], 'mmse_norm'] = -1.0  # Sentinel for missing

    n_valid_mmse = df['mmse_valid'].sum()
    print(f'\nMMSE: {n_valid_mmse}/{len(df)} patients have valid scores')

    return df


def center_crop_3d(volume, target_shape):
    """Center crop a 3D volume to target_shape."""
    d, h, w = volume.shape
    td, th, tw = target_shape
    sd = max(0, (d - td) // 2)
    sh = max(0, (h - th) // 2)
    sw = max(0, (w - tw) // 2)
    cropped = volume[sd:sd+td, sh:sh+th, sw:sw+tw]
    # Pad if smaller than target
    if cropped.shape != target_shape:
        padded = np.full(target_shape, -1.0, dtype=np.float32)
        pd_ = (td - cropped.shape[0]) // 2
        ph_ = (th - cropped.shape[1]) // 2
        pw_ = (tw - cropped.shape[2]) // 2
        padded[pd_:pd_+cropped.shape[0],
               ph_:ph_+cropped.shape[1],
               pw_:pw_+cropped.shape[2]] = cropped
        return padded
    return cropped


def augment_volume_pair(mri, pet, cfg):
    """Apply matched augmentations to MRI and PET volumes."""
    # Random horizontal flip
    if random.random() < cfg.AUG_FLIP_P:
        mri = np.flip(mri, axis=2).copy()
        pet = np.flip(pet, axis=2).copy()

    # Gaussian noise (MRI only, PET stays clean)
    if cfg.AUG_NOISE_STD > 0 and random.random() < cfg.AUG_NOISE_P:
        noise = np.random.normal(0, cfg.AUG_NOISE_STD, mri.shape).astype(np.float32)
        mri = np.clip(mri + noise, -1.0, 1.0)

    return mri, pet


def _find_pet_volume(naccid, data_root, cfg):
    """Locate PET volume for a patient, preferring real over synthetic.
    Returns (pet_path, is_synthetic) or (None, None) if not found.
    """
    # 1. Real PET in main data root
    real_path = os.path.join(data_root, naccid, cfg.REAL_PET_FILE)
    if os.path.exists(real_path):
        return real_path, False

    # 2. Synthetic PET in main data root
    synth_in_main = os.path.join(data_root, naccid, cfg.SYNTHETIC_PET_FILE)
    if os.path.exists(synth_in_main):
        return synth_in_main, True

    # 3. Synthetic PET in separate synthetic root
    if cfg.SYNTHETIC_PET_ROOT:
        synth_path = os.path.join(cfg.SYNTHETIC_PET_ROOT, naccid, cfg.SYNTHETIC_PET_FILE)
        if os.path.exists(synth_path):
            return synth_path, True
        # Also check if synthetic root uses the same filename as real
        synth_path2 = os.path.join(cfg.SYNTHETIC_PET_ROOT, naccid, cfg.REAL_PET_FILE)
        if os.path.exists(synth_path2):
            return synth_path2, True

    return None, None


class ADClassificationDataset(Dataset):
    """3D MRI + PET volume dataset for AD classification.
    Tracks whether each sample uses real or synthetic PET,
    and provides a per-sample confidence weight for the PET modality.
    """

    def __init__(self, patient_df, data_root, cfg, augment=False):
        self.cfg = cfg
        self.augment = augment
        # (mri_path, pet_path, label, mmse_norm, mmse_valid, pet_confidence)
        self.samples = []

        found_real, found_synth, missing = 0, 0, 0
        for _, row in patient_df.iterrows():
            naccid = row['NACCID']
            mri_p = os.path.join(data_root, naccid, 'mri_processed.npy')
            if not os.path.exists(mri_p):
                missing += 1
                continue

            pet_p, is_synthetic = _find_pet_volume(naccid, data_root, cfg)
            if pet_p is None:
                missing += 1
                continue

            # Assign PET confidence based on source
            pet_conf = cfg.SYNTH_PET_CONFIDENCE if is_synthetic else cfg.REAL_PET_CONFIDENCE

            self.samples.append((
                mri_p, pet_p,
                int(row['label']),
                float(row['mmse_norm']),
                bool(row['mmse_valid']),
                float(pet_conf),
            ))
            if is_synthetic:
                found_synth += 1
            else:
                found_real += 1

        print(f'  Dataset: {found_real} real PET + {found_synth} synthetic PET, '
              f'{missing} missing')

    def __len__(self):
        return len(self.samples)

    def get_labels(self):
        """Return all labels for weighted sampling."""
        return [s[2] for s in self.samples]

    def __getitem__(self, idx):
        mri_p, pet_p, label, mmse, mmse_valid, pet_conf = self.samples[idx]

        mri = np.load(mri_p, mmap_mode='r').astype(np.float32)
        pet = np.load(pet_p, mmap_mode='r').astype(np.float32)

        # Center crop
        mri = center_crop_3d(np.array(mri), self.cfg.CROP_SHAPE)
        pet = center_crop_3d(np.array(pet), self.cfg.CROP_SHAPE)

        # Augmentation
        if self.augment:
            mri, pet = augment_volume_pair(mri.copy(), pet.copy(), self.cfg)

        # Add channel dim: (D, H, W) -> (1, D, H, W)
        mri_t = torch.from_numpy(mri).unsqueeze(0)
        pet_t = torch.from_numpy(pet).unsqueeze(0)

        return {
            'mri': mri_t,
            'pet': pet_t,
            'label': torch.tensor(label, dtype=torch.long),
            'mmse': torch.tensor(mmse, dtype=torch.float32),
            'mmse_valid': torch.tensor(mmse_valid, dtype=torch.bool),
            'pet_confidence': torch.tensor(pet_conf, dtype=torch.float32),
        }


# ---- Build datasets ----
print('Loading patient labels...')
patient_df = load_patient_labels(cfg.CSV_PATH, cfg.LABEL_MAP)

# Stratified split: train / val / test
from sklearn.model_selection import train_test_split

train_val_df, test_df = train_test_split(
    patient_df, test_size=cfg.TEST_SPLIT,
    stratify=patient_df['label'], random_state=SEED)

val_ratio = cfg.VAL_SPLIT / (1.0 - cfg.TEST_SPLIT)
train_df, val_df = train_test_split(
    train_val_df, test_size=val_ratio,
    stratify=train_val_df['label'], random_state=SEED)

print(f'\nSplit: {len(train_df)} train / {len(val_df)} val / {len(test_df)} test')

train_dataset = ADClassificationDataset(train_df, cfg.DATA_ROOT, cfg, augment=True)
val_dataset   = ADClassificationDataset(val_df,   cfg.DATA_ROOT, cfg, augment=False)
test_dataset  = ADClassificationDataset(test_df,  cfg.DATA_ROOT, cfg, augment=False)

# Class-weighted sampler for imbalanced training data
train_labels = train_dataset.get_labels()
class_counts = Counter(train_labels)
class_weights = {c: 1.0 / count for c, count in class_counts.items()}
sample_weights = [class_weights[l] for l in train_labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE,
                          sampler=sampler, num_workers=2,
                          pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE,
                        shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=cfg.BATCH_SIZE,
                         shuffle=False, num_workers=2, pin_memory=True)

print(f'\nTrain: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')
print(f'Class weights: {class_weights}')


In [ ]:
# ============================================================
# VISUALIZE CLASS DISTRIBUTION + SAMPLE DATA
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution
labels_all = patient_df['label'].values
counts = [np.sum(labels_all == i) for i in range(cfg.NUM_CLASSES)]
colors = ['#2ecc71', '#f39c12', '#e74c3c']
axes[0].bar(cfg.CLASS_NAMES, counts, color=colors, edgecolor='black')
axes[0].set_ylabel('Count')
axes[0].set_title('Class Distribution (All Patients)')
for i, c in enumerate(counts):
    axes[0].text(i, c + 1, str(c), ha='center', fontweight='bold')

# MMSE distribution by class
valid_mmse = patient_df[patient_df['mmse_valid']]
for i, name in enumerate(cfg.CLASS_NAMES):
    subset = valid_mmse[valid_mmse['label'] == i]['NACCMMSE']
    if len(subset) > 0:
        axes[1].hist(subset, bins=15, alpha=0.5, label=f'{name} (n={len(subset)})',
                     color=colors[i], edgecolor='black')
axes[1].set_xlabel('MMSE Score')
axes[1].set_ylabel('Count')
axes[1].set_title('MMSE Distribution by Class')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(cfg.SAVE_DIR, 'data_distribution.png'), dpi=150)
plt.show()


## 2. Model Architecture

### Dual-Branch 3D ResNet-18 + Asymmetric Cross-Attention Fusion

| Component | Detail |
|---|---|
| **MRI Branch** | 3D ResNet-18 (1ch -> 512-dim features, spatial 3x4x3) |
| **PET Branch** | 3D ResNet-18 (1ch -> 512-dim features, spatial 3x4x3) |
| **Cross-Attention** | 2-layer Transformer with 8 heads, asymmetric gating |
| **Classification Head** | Linear(512, 256) -> ReLU -> Dropout -> Linear(256, 3) |
| **Regression Head** | Linear(512, 128) -> ReLU -> Dropout -> Linear(128, 1) |

The **asymmetric** aspect: MRI and PET branches have separate weights,
and a learnable gate controls the relative importance of MRI-attended-to-PET
vs PET-attended-to-MRI features. This allows the model to learn which
modality is more informative for each spatial region.


In [ ]:
# ============================================================
# 3D RESNET-18 FEATURE EXTRACTOR
# ============================================================

class BasicBlock3D(nn.Module):
    """Basic residual block for 3D ResNet."""
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv3d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm3d(planes)
        self.conv2 = nn.Conv3d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm3d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv3d(in_planes, planes, 1, stride=stride, bias=False),
                nn.BatchNorm3d(planes),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)


class ResNet3D(nn.Module):
    """3D ResNet-18 backbone for single-channel volumetric input.
    Returns both the pooled feature vector and spatial feature maps (for Grad-CAM).
    """

    def __init__(self, in_channels=1, num_blocks=(2, 2, 2, 2)):
        super().__init__()
        self.in_planes = 64

        self.conv1 = nn.Conv3d(in_channels, 64, 7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm3d(64)
        self.maxpool = nn.MaxPool3d(3, stride=2, padding=1)

        self.layer1 = self._make_layer(64,  num_blocks[0], stride=1)
        self.layer2 = self._make_layer(128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(512, num_blocks[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool3d(1)

        # Initialize weights
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm3d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, planes, num_blocks, stride):
        layers = [BasicBlock3D(self.in_planes, planes, stride)]
        self.in_planes = planes
        for _ in range(1, num_blocks):
            layers.append(BasicBlock3D(planes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        """Returns (pooled_features, spatial_features).
        pooled: (B, 512), spatial: (B, 512, D', H', W')
        """
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        spatial = self.layer4(x)  # (B, 512, D', H', W') - for Grad-CAM
        pooled = self.avgpool(spatial).flatten(1)  # (B, 512)
        return pooled, spatial


# Quick shape check
_test = torch.randn(1, 1, *cfg.CROP_SHAPE)
_net = ResNet3D(in_channels=1)
_pooled, _spatial = _net(_test)
print(f'Input:   {_test.shape}')
print(f'Spatial: {_spatial.shape}')  # For cross-attention
print(f'Pooled:  {_pooled.shape}')   # Feature vector
print(f'ResNet3D params: {sum(p.numel() for p in _net.parameters()):,}')
del _test, _net, _pooled, _spatial; gc.collect()


In [ ]:
# ============================================================
# ASYMMETRIC CROSS-ATTENTION FUSION MODULE
# ============================================================

class AsymmetricCrossAttentionLayer(nn.Module):
    """Single layer of asymmetric cross-attention between two modalities.
    MRI queries attend to PET keys/values and vice versa,
    with a learnable gate controlling the fusion ratio.

    Accepts a per-sample `pet_confidence` in [0, 1] that controls
    how much the PET branch contributes relative to MRI:
      - pet_confidence=1.0 (real PET): gate learns freely
      - pet_confidence=0.3 (synthetic PET): gate is biased toward MRI
        so the fused output is ~70% MRI-attended, ~30% PET-attended
    """

    def __init__(self, dim, num_heads=8, dropout=0.1):
        super().__init__()
        # MRI -> PET cross-attention
        self.mri_to_pet = nn.MultiheadAttention(
            dim, num_heads, dropout=dropout, batch_first=True)
        # PET -> MRI cross-attention
        self.pet_to_mri = nn.MultiheadAttention(
            dim, num_heads, dropout=dropout, batch_first=True)

        self.norm_mri = nn.LayerNorm(dim)
        self.norm_pet = nn.LayerNorm(dim)

        # Asymmetric gating: learned balance between modalities
        self.gate = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.Sigmoid(),
        )

        # Feed-forward network
        self.ffn = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 4, dim),
            nn.Dropout(dropout),
        )
        self.norm_out = nn.LayerNorm(dim)

    def forward(self, mri_tokens, pet_tokens, pet_confidence=None):
        """Args:
            mri_tokens:     (B, N, D)
            pet_tokens:     (B, N, D)
            pet_confidence: (B,) float in [0, 1], or None for balanced fusion
        Returns:
            fused_tokens:   (B, N, D)
            attn_weights:   (B, N, N)
        """
        # Scale PET tokens by confidence BEFORE cross-attention.
        # When pet_confidence=0.3, PET keys/values are attenuated,
        # so MRI-queries-attending-to-PET will extract less information.
        if pet_confidence is not None:
            pc = pet_confidence.view(-1, 1, 1)  # (B, 1, 1)
            scaled_pet = pet_tokens * pc
        else:
            scaled_pet = pet_tokens

        # Cross-attention: MRI queries attend to (scaled) PET
        mri_attended, _ = self.mri_to_pet(
            query=mri_tokens, key=scaled_pet, value=scaled_pet)
        mri_attended = self.norm_mri(mri_tokens + mri_attended)

        # Cross-attention: PET queries attend to MRI (MRI is always full-strength)
        pet_attended, attn_w = self.pet_to_mri(
            query=scaled_pet, key=mri_tokens, value=mri_tokens)
        pet_attended = self.norm_pet(scaled_pet + pet_attended)

        # Asymmetric gated fusion, biased by pet_confidence.
        # raw_gate near 0.5 when balanced; we bias it toward MRI
        # when pet_confidence < 1.0 so MRI dominates.
        mri_pool = mri_attended.mean(dim=1)  # (B, D)
        pet_pool = pet_attended.mean(dim=1)   # (B, D)
        raw_gate = self.gate(torch.cat([mri_pool, pet_pool], dim=-1))  # (B, D) in [0,1]

        if pet_confidence is not None:
            # Remap gate: when pet_confidence=0.3, push gate toward 1.0 (= more MRI)
            # gate_bias = 1.0 - pet_confidence, so for synth: bias=0.7
            # final_gate = raw_gate * pet_confidence + (1 - pet_confidence)
            # This means: if pet_conf=0.3, gate is at least 0.7 (MRI weight >= 70%)
            pc = pet_confidence.view(-1, 1)  # (B, 1)
            gate = raw_gate * pc + (1.0 - pc)
        else:
            gate = raw_gate

        # gate -> 1.0 means all MRI-attended, 0.0 means all PET-attended
        fused = gate.unsqueeze(1) * mri_attended + (1 - gate.unsqueeze(1)) * pet_attended

        # FFN + residual
        fused = self.norm_out(fused + self.ffn(fused))

        return fused, attn_w


class AsymmetricCrossAttentionFusion(nn.Module):
    """Multi-layer cross-attention fusion for MRI and PET spatial features.
    Propagates pet_confidence through all layers to maintain modality weighting.
    """

    def __init__(self, feature_dim=512, num_heads=8, num_layers=2, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            AsymmetricCrossAttentionLayer(feature_dim, num_heads, dropout)
            for _ in range(num_layers)
        ])
        self.pool = nn.AdaptiveAvgPool1d(1)

    def forward(self, mri_spatial, pet_spatial, pet_confidence=None):
        """Args:
            mri_spatial:    (B, C, D, H, W) from ResNet layer4
            pet_spatial:    (B, C, D, H, W) from ResNet layer4
            pet_confidence: (B,) float in [0, 1], or None
        Returns:
            fused_features: (B, C)
            attn_weights:   from last layer
        """
        B, C = mri_spatial.shape[:2]
        # Flatten spatial dims to sequence: (B, C, D*H*W) -> (B, N, C)
        mri_tokens = mri_spatial.flatten(2).permute(0, 2, 1)  # (B, N, C)
        pet_tokens = pet_spatial.flatten(2).permute(0, 2, 1)  # (B, N, C)

        attn_weights = None
        for layer in self.layers:
            mri_tokens, attn_weights = layer(mri_tokens, pet_tokens, pet_confidence)
            pet_tokens = mri_tokens  # Feed fused into next layer

        # Pool sequence to single vector
        fused = mri_tokens.permute(0, 2, 1)  # (B, C, N)
        fused = self.pool(fused).squeeze(-1)  # (B, C)

        return fused, attn_weights


print('Cross-attention fusion module defined (with modality confidence weighting).')


In [ ]:
# ============================================================
# FULL MULTI-TASK CLASSIFICATION MODEL
# ============================================================

class ADClassificationModel(nn.Module):
    """Dual-branch 3D ResNet + Cross-Attention Fusion + Multi-Task Heads.

    Supports:
    - Modality confidence weighting (pet_confidence) for real vs synthetic PET
    - MC Dropout at inference for uncertainty estimation
    """

    def __init__(self, num_classes=3, feature_dim=512, dropout=0.3,
                 fusion_heads=8, fusion_layers=2, fusion_dropout=0.1):
        super().__init__()

        # Dual 3D ResNet backbones (separate weights)
        self.mri_backbone = ResNet3D(in_channels=1)
        self.pet_backbone = ResNet3D(in_channels=1)

        # Asymmetric cross-attention fusion
        self.fusion = AsymmetricCrossAttentionFusion(
            feature_dim=feature_dim, num_heads=fusion_heads,
            num_layers=fusion_layers, dropout=fusion_dropout)

        # Classification head (3-class: CN, MCI, AD)
        self.cls_head = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

        # Regression head (MMSE score prediction)
        self.reg_head = nn.Sequential(
            nn.Linear(feature_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
            nn.Sigmoid(),  # Output in [0, 1] (normalized MMSE)
        )

    def forward(self, mri, pet, pet_confidence=None):
        """Forward pass with modality confidence weighting.
        Args:
            mri:            (B, 1, D, H, W) MRI volume
            pet:            (B, 1, D, H, W) PET volume (real or synthetic)
            pet_confidence: (B,) float in [0, 1] — how much to trust PET.
                            1.0 = real PET (full trust), 0.3 = synthetic (limited).
                            None = treat all PET as fully trusted (backward compat).
        Returns:
            cls_logits:   (B, num_classes)
            mmse_pred:    (B, 1)
            attn_weights: attention weights for visualization
        """
        # Extract features from both modalities
        mri_pooled, mri_spatial = self.mri_backbone(mri)
        pet_pooled, pet_spatial = self.pet_backbone(pet)

        # Cross-attention fusion with confidence weighting
        fused, attn_weights = self.fusion(mri_spatial, pet_spatial, pet_confidence)

        # Multi-task heads
        cls_logits = self.cls_head(fused)
        mmse_pred = self.reg_head(fused)

        return cls_logits, mmse_pred, attn_weights

    def enable_mc_dropout(self):
        """Enable dropout layers for MC Dropout inference."""
        for m in self.modules():
            if isinstance(m, nn.Dropout):
                m.train()


def build_model(cfg):
    return ADClassificationModel(
        num_classes=cfg.NUM_CLASSES,
        feature_dim=cfg.FEATURE_DIM,
        dropout=cfg.DROPOUT,
        fusion_heads=cfg.FUSION_HEADS,
        fusion_layers=cfg.FUSION_LAYERS,
        fusion_dropout=cfg.FUSION_DROPOUT,
    )


# Model summary
_model = build_model(cfg)
n_params = sum(p.numel() for p in _model.parameters())
print(f'Total parameters:     {n_params:,}')
print(f'MRI backbone:         {sum(p.numel() for p in _model.mri_backbone.parameters()):,}')
print(f'PET backbone:         {sum(p.numel() for p in _model.pet_backbone.parameters()):,}')
print(f'Cross-attention:      {sum(p.numel() for p in _model.fusion.parameters()):,}')
print(f'Classification head:  {sum(p.numel() for p in _model.cls_head.parameters()):,}')
print(f'Regression head:      {sum(p.numel() for p in _model.reg_head.parameters()):,}')
print(f'Model size:           {n_params * 4 / 1e6:.0f} MB (FP32)')
print(f'\nModality weighting:')
print(f'  Real PET confidence:      {cfg.REAL_PET_CONFIDENCE}')
print(f'  Synthetic PET confidence: {cfg.SYNTH_PET_CONFIDENCE}')
print(f'  -> Synthetic: MRI ~{(1-cfg.SYNTH_PET_CONFIDENCE)*100:.0f}%, PET ~{cfg.SYNTH_PET_CONFIDENCE*100:.0f}%')
del _model; gc.collect()


## 3. Training

### Multi-Task Loss
$$\mathcal{L}_{total} = \lambda_{cls} \cdot \mathcal{L}_{CE} + \lambda_{reg} \cdot \mathcal{L}_{SmoothL1}$$

- **Classification**: Cross-entropy with inverse-frequency class weights
- **Regression**: Smooth L1 loss on MMSE (only for patients with valid scores)
- **Optimizer**: AdamW (lr=1e-4, weight_decay=1e-4)
- **Schedule**: Cosine annealing with linear warmup
- **Mixed Precision**: FP16 on CUDA


In [ ]:
# ============================================================
# LOSS FUNCTIONS + TRAINING UTILITIES
# ============================================================

def compute_class_weights(dataset, num_classes, device):
    """Compute inverse-frequency class weights for cross-entropy."""
    labels = dataset.get_labels()
    counts = Counter(labels)
    total = sum(counts.values())
    weights = torch.zeros(num_classes)
    for c in range(num_classes):
        weights[c] = total / (num_classes * max(1, counts.get(c, 1)))
    return weights.to(device)


def get_lr_lambda(epoch, warmup_epochs, total_epochs):
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


def train_one_epoch(model, loader, optimizer, cls_criterion, cfg, scaler=None):
    model.train()
    total_loss, cls_loss_sum, reg_loss_sum = 0.0, 0.0, 0.0
    correct, total_samples = 0, 0

    for batch in tqdm(loader, desc='Train', leave=False):
        mri = batch['mri'].to(device, non_blocking=True)
        pet = batch['pet'].to(device, non_blocking=True)
        labels = batch['label'].to(device, non_blocking=True)
        mmse = batch['mmse'].to(device, non_blocking=True)
        mmse_valid = batch['mmse_valid'].to(device, non_blocking=True)
        pet_conf = batch['pet_confidence'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if scaler is not None:
            with autocast('cuda'):
                cls_logits, mmse_pred, _ = model(mri, pet, pet_conf)
                l_cls = cls_criterion(cls_logits, labels)
                # Regression loss only for valid MMSE
                if mmse_valid.any():
                    l_reg = F.smooth_l1_loss(
                        mmse_pred[mmse_valid].squeeze(-1),
                        mmse[mmse_valid])
                else:
                    l_reg = torch.tensor(0.0, device=device)
                loss = cfg.CLS_WEIGHT * l_cls + cfg.REG_WEIGHT * l_reg
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            cls_logits, mmse_pred, _ = model(mri, pet, pet_conf)
            l_cls = cls_criterion(cls_logits, labels)
            if mmse_valid.any():
                l_reg = F.smooth_l1_loss(
                    mmse_pred[mmse_valid].squeeze(-1),
                    mmse[mmse_valid])
            else:
                l_reg = torch.tensor(0.0, device=device)
            loss = cfg.CLS_WEIGHT * l_cls + cfg.REG_WEIGHT * l_reg
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            optimizer.step()

        total_loss += loss.item()
        cls_loss_sum += l_cls.item()
        reg_loss_sum += l_reg.item()
        preds = cls_logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    n = len(loader)
    return {
        'loss': total_loss / n,
        'cls_loss': cls_loss_sum / n,
        'reg_loss': reg_loss_sum / n,
        'accuracy': correct / max(1, total_samples),
    }


@torch.no_grad()
def evaluate(model, loader, cls_criterion, cfg):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    all_mmse_pred, all_mmse_true, all_mmse_valid = [], [], []
    total_loss = 0.0

    for batch in tqdm(loader, desc='Eval', leave=False):
        mri = batch['mri'].to(device, non_blocking=True)
        pet = batch['pet'].to(device, non_blocking=True)
        labels = batch['label'].to(device, non_blocking=True)
        mmse = batch['mmse'].to(device, non_blocking=True)
        mmse_valid = batch['mmse_valid'].to(device, non_blocking=True)
        pet_conf = batch['pet_confidence'].to(device, non_blocking=True)

        with autocast('cuda', enabled=(device.type == 'cuda')):
            cls_logits, mmse_pred, _ = model(mri, pet, pet_conf)
            l_cls = cls_criterion(cls_logits, labels)
            total_loss += l_cls.item()

        probs = F.softmax(cls_logits, dim=1)
        preds = cls_logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.append(probs.cpu().numpy())
        all_mmse_pred.extend(mmse_pred.squeeze(-1).cpu().numpy())
        all_mmse_true.extend(mmse.cpu().numpy())
        all_mmse_valid.extend(mmse_valid.cpu().numpy())

    all_probs = np.concatenate(all_probs, axis=0)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_mmse_pred = np.array(all_mmse_pred)
    all_mmse_true = np.array(all_mmse_true)
    all_mmse_valid = np.array(all_mmse_valid, dtype=bool)

    acc = accuracy_score(all_labels, all_preds)
    bal_acc = balanced_accuracy_score(all_labels, all_preds)

    # MMSE metrics (only valid scores)
    mmse_mae = -1.0
    if all_mmse_valid.any():
        # Denormalize: multiply by 30
        pred_mmse = all_mmse_pred[all_mmse_valid] * 30.0
        true_mmse = all_mmse_true[all_mmse_valid] * 30.0
        mmse_mae = mean_absolute_error(true_mmse, pred_mmse)

    return {
        'loss': total_loss / len(loader),
        'accuracy': acc,
        'balanced_accuracy': bal_acc,
        'mmse_mae': mmse_mae,
        'preds': all_preds,
        'labels': all_labels,
        'probs': all_probs,
    }


print('Training utilities defined.')


In [ ]:
# ============================================================
# MAIN TRAINING LOOP
# ============================================================

def run_training(cfg, train_loader, val_loader, resume_path=None):
    model = build_model(cfg).to(device)
    cls_weights = compute_class_weights(train_dataset, cfg.NUM_CLASSES, device)
    cls_criterion = nn.CrossEntropyLoss(weight=cls_weights)
    print(f'Class weights: {cls_weights.cpu().tolist()}')

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda ep: get_lr_lambda(ep, cfg.WARMUP_EPOCHS, cfg.EPOCHS))
    scaler = GradScaler('cuda') if device.type == 'cuda' else None

    best_bal_acc = 0.0
    start_epoch = 1
    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_bal_acc': [], 'val_epochs': []}

    # Resume
    if resume_path and os.path.exists(resume_path):
        ckpt = torch.load(resume_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        start_epoch = ckpt.get('epoch', 0) + 1
        best_bal_acc = ckpt.get('best_bal_acc', 0.0)
        history = ckpt.get('history', history)
        # Ensure val_epochs key exists for older checkpoints
        if 'val_epochs' not in history:
            history['val_epochs'] = []
        if 'scheduler_state_dict' in ckpt:
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        print(f'Resumed from epoch {start_epoch - 1}, best bal_acc: {best_bal_acc:.4f}')

    n_params = sum(p.numel() for p in model.parameters())
    print(f'Model: {n_params:,} params | Epochs {start_epoch}-{cfg.EPOCHS}')
    print('=' * 70)

    for epoch in range(start_epoch, cfg.EPOCHS + 1):
        t0 = time.time()
        train_metrics = train_one_epoch(
            model, train_loader, optimizer, cls_criterion, cfg, scaler)
        scheduler.step()

        history['train_loss'].append(train_metrics['loss'])

        lr_now = optimizer.param_groups[0]['lr']
        elapsed = time.time() - t0
        print(f'Epoch {epoch:3d}/{cfg.EPOCHS} | '
              f'Loss: {train_metrics["loss"]:.4f} '
              f'(cls:{train_metrics["cls_loss"]:.4f} reg:{train_metrics["reg_loss"]:.4f}) | '
              f'Acc: {train_metrics["accuracy"]:.3f} | '
              f'LR: {lr_now:.2e} | {elapsed:.0f}s')

        # Validation
        if epoch % cfg.VAL_EVERY == 0 or epoch == cfg.EPOCHS:
            val_metrics = evaluate(model, val_loader, cls_criterion, cfg)
            history['val_loss'].append(val_metrics['loss'])
            history['val_acc'].append(val_metrics['accuracy'])
            history['val_bal_acc'].append(val_metrics['balanced_accuracy'])
            history['val_epochs'].append(epoch)

            print(f'  >> Val Acc: {val_metrics["accuracy"]:.4f} | '
                  f'Bal Acc: {val_metrics["balanced_accuracy"]:.4f} | '
                  f'MMSE MAE: {val_metrics["mmse_mae"]:.2f}')

            if val_metrics['balanced_accuracy'] > best_bal_acc:
                best_bal_acc = val_metrics['balanced_accuracy']
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'best_bal_acc': best_bal_acc,
                    'history': history,
                }, os.path.join(cfg.SAVE_DIR, 'best_model.pt'))
                print(f'  >> NEW BEST (bal_acc={best_bal_acc:.4f})')

        # Checkpoint
        if epoch % cfg.SAVE_EVERY == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_bal_acc': best_bal_acc,
                'history': history,
            }, os.path.join(cfg.SAVE_DIR, 'latest_checkpoint.pt'))

    print('=' * 70)
    print(f'Training complete. Best balanced accuracy: {best_bal_acc:.4f}')
    return model, history, best_bal_acc


# ---- Run Training ----
resume = cfg.RESUME_PATH or os.path.join(cfg.SAVE_DIR, 'latest_checkpoint.pt')
if not os.path.exists(resume):
    resume = None

model, history, best_bal_acc = run_training(
    cfg, train_loader, val_loader, resume_path=resume)

In [ ]:
# ============================================================
# LOAD BEST MODEL + TRAINING CURVES
# ============================================================

ckpt_path = os.path.join(cfg.SAVE_DIR, 'best_model.pt')
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model = build_model(cfg).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    history = ckpt.get('history', history)
    print(f'Loaded best model from epoch {ckpt["epoch"]}')

model.eval()

# Reconstruct validation epoch indices robustly
n_val = len(history.get('val_loss', []))
if 'val_epochs' in history and len(history['val_epochs']) == n_val:
    val_epochs = history['val_epochs']
elif n_val > 0:
    # Fallback: reconstruct from config (multiples of VAL_EVERY, plus final epoch)
    n_train = len(history['train_loss'])
    val_epochs = list(range(cfg.VAL_EVERY, n_train + 1, cfg.VAL_EVERY))
    # Add final epoch if it wasn't a multiple of VAL_EVERY
    if n_train > 0 and n_train % cfg.VAL_EVERY != 0:
        val_epochs.append(n_train)
    # Truncate or pad to match actual val data length
    val_epochs = val_epochs[:n_val]
    # If still too short, use simple sequential indices
    if len(val_epochs) < n_val:
        val_epochs = list(range(1, n_val + 1))
else:
    val_epochs = []

# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(range(1, len(history['train_loss']) + 1), history['train_loss'],
             label='Train', linewidth=0.8)
if n_val > 0:
    axes[0].plot(val_epochs, history['val_loss'], 'o-', label='Val', markersize=3)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

if n_val > 0:
    axes[1].plot(val_epochs, history['val_acc'], 'o-', label='Accuracy', markersize=3)
    axes[1].plot(val_epochs, history['val_bal_acc'], 's--', label='Balanced Acc', markersize=3)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Validation Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Per-class performance over training
axes[2].text(0.5, 0.5, f'Best Balanced Accuracy:\n{best_bal_acc:.4f}',
             ha='center', va='center', fontsize=20, fontweight='bold',
             transform=axes[2].transAxes)
axes[2].set_title('Best Result'); axes[2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(cfg.SAVE_DIR, 'training_curves.png'), dpi=150)
plt.show()

## 4. Uncertainty Estimation

### MC Dropout (Gal & Ghahramani, 2016)
Epistemic uncertainty via multiple stochastic forward passes with dropout active.
- Run N=30 forward passes per sample
- Mean softmax probabilities = calibrated prediction
- Predictive entropy = uncertainty measure

### Temperature Scaling (Guo et al., 2017)
Post-hoc calibration of softmax probabilities:
- Learn a single temperature T on validation set
- Calibrated logits = raw logits / T
- Measured by Expected Calibration Error (ECE)


In [ ]:
# ============================================================
# MC DROPOUT UNCERTAINTY ESTIMATION
# ============================================================

@torch.no_grad()
def mc_dropout_inference(model, loader, cfg, n_samples=30):
    """Run MC Dropout inference for uncertainty estimation.
    Returns per-sample: mean probs, predictions, uncertainty, labels.
    """
    model.eval()
    model.enable_mc_dropout()  # Keep dropout active

    all_mc_probs = []  # (N_data, n_samples, num_classes)
    all_labels = []
    all_mc_mmse = []
    all_mmse_true = []
    all_mmse_valid = []

    for batch in tqdm(loader, desc='MC Dropout'):
        mri = batch['mri'].to(device, non_blocking=True)
        pet = batch['pet'].to(device, non_blocking=True)
        labels = batch['label']
        mmse = batch['mmse']
        mmse_valid = batch['mmse_valid']
        pet_conf = batch['pet_confidence'].to(device, non_blocking=True)

        batch_probs = []
        batch_mmse = []

        for _ in range(n_samples):
            with autocast('cuda', enabled=(device.type == 'cuda')):
                cls_logits, mmse_pred, _ = model(mri, pet, pet_conf)
            probs = F.softmax(cls_logits, dim=1)
            batch_probs.append(probs.cpu().numpy())
            batch_mmse.append(mmse_pred.squeeze(-1).cpu().numpy())

        # Stack: (n_samples, B, C) -> (B, n_samples, C)
        batch_probs = np.stack(batch_probs, axis=1)
        batch_mmse = np.stack(batch_mmse, axis=1)

        all_mc_probs.append(batch_probs)
        all_labels.extend(labels.numpy())
        all_mc_mmse.append(batch_mmse)
        all_mmse_true.extend(mmse.numpy())
        all_mmse_valid.extend(mmse_valid.numpy())

    model.eval()  # Restore normal eval mode

    all_mc_probs = np.concatenate(all_mc_probs, axis=0)  # (N, n_samples, C)
    all_mc_mmse = np.concatenate(all_mc_mmse, axis=0)    # (N, n_samples)
    all_labels = np.array(all_labels)
    all_mmse_true = np.array(all_mmse_true)
    all_mmse_valid = np.array(all_mmse_valid, dtype=bool)

    # Mean prediction
    mean_probs = all_mc_probs.mean(axis=1)  # (N, C)
    predictions = mean_probs.argmax(axis=1)

    # Predictive entropy (uncertainty)
    entropy = -np.sum(mean_probs * np.log(mean_probs + 1e-10), axis=1)

    # Mutual information (epistemic uncertainty)
    expected_entropy = -np.mean(
        np.sum(all_mc_probs * np.log(all_mc_probs + 1e-10), axis=2), axis=1)
    mutual_info = entropy - expected_entropy

    # MMSE uncertainty
    mmse_mean = all_mc_mmse.mean(axis=1)
    mmse_std = all_mc_mmse.std(axis=1)

    return {
        'mc_probs': all_mc_probs,
        'mean_probs': mean_probs,
        'predictions': predictions,
        'labels': all_labels,
        'entropy': entropy,
        'mutual_info': mutual_info,
        'mmse_mean': mmse_mean,
        'mmse_std': mmse_std,
        'mmse_true': all_mmse_true,
        'mmse_valid': all_mmse_valid,
    }


# ============================================================
# TEMPERATURE SCALING + ECE
# ============================================================

class TemperatureScaling(nn.Module):
    """Post-hoc temperature scaling for calibration."""

    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        return logits / self.temperature


def compute_ece(probs, labels, n_bins=15):
    """Expected Calibration Error."""
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracies = (predictions == labels).astype(float)

    bin_boundaries = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    bin_data = []

    for i in range(n_bins):
        mask = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i + 1])
        if mask.sum() > 0:
            bin_acc = accuracies[mask].mean()
            bin_conf = confidences[mask].mean()
            bin_size = mask.sum()
            ece += (bin_size / len(labels)) * abs(bin_acc - bin_conf)
            bin_data.append((bin_conf, bin_acc, bin_size))

    return ece, bin_data


def fit_temperature(model, val_loader, cfg):
    """Fit temperature parameter on validation set."""
    model.eval()
    all_logits, all_labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            mri = batch['mri'].to(device)
            pet = batch['pet'].to(device)
            labels = batch['label']
            pet_conf = batch['pet_confidence'].to(device)
            with autocast('cuda', enabled=(device.type == 'cuda')):
                cls_logits, _, _ = model(mri, pet, pet_conf)
            all_logits.append(cls_logits.cpu())
            all_labels.append(labels)

    logits = torch.cat(all_logits, dim=0)
    labels = torch.cat(all_labels, dim=0)

    # Before calibration
    probs_before = F.softmax(logits, dim=1).numpy()
    ece_before, _ = compute_ece(probs_before, labels.numpy())
    print(f'ECE before calibration: {ece_before:.4f}')

    # Fit temperature
    temp_model = TemperatureScaling().to(device)
    temp_optimizer = torch.optim.LBFGS(
        [temp_model.temperature], lr=cfg.TEMP_LR, max_iter=cfg.TEMP_ITERS)
    nll_criterion = nn.CrossEntropyLoss()
    logits_d = logits.to(device)
    labels_d = labels.to(device)

    def closure():
        temp_optimizer.zero_grad()
        loss = nll_criterion(temp_model(logits_d), labels_d)
        loss.backward()
        return loss

    temp_optimizer.step(closure)

    # After calibration
    with torch.no_grad():
        calibrated_logits = temp_model(logits_d)
        probs_after = F.softmax(calibrated_logits, dim=1).cpu().numpy()

    ece_after, _ = compute_ece(probs_after, labels.numpy())
    temp_val = temp_model.temperature.item()
    print(f'ECE after calibration:  {ece_after:.4f}')
    print(f'Learned temperature:    {temp_val:.4f}')

    return temp_model, ece_before, ece_after


# ---- Run MC Dropout on test set ----
print('Running MC Dropout inference on test set...')
mc_results = mc_dropout_inference(model, test_loader, cfg, n_samples=cfg.MC_SAMPLES)

mc_acc = accuracy_score(mc_results['labels'], mc_results['predictions'])
mc_bal_acc = balanced_accuracy_score(mc_results['labels'], mc_results['predictions'])
print(f'\nMC Dropout Test Accuracy:          {mc_acc:.4f}')
print(f'MC Dropout Test Balanced Accuracy:  {mc_bal_acc:.4f}')
print(f'Mean predictive entropy:            {mc_results["entropy"].mean():.4f}')
print(f'Mean mutual information:            {mc_results["mutual_info"].mean():.4f}')

# ---- Fit Temperature Scaling ----
print('\nFitting temperature scaling on validation set...')
temp_model, ece_before, ece_after = fit_temperature(model, val_loader, cfg)


In [ ]:
# ============================================================
# UNCERTAINTY VISUALIZATION
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

colors = ['#2ecc71', '#f39c12', '#e74c3c']

# 1. Entropy distribution by correctness
correct = mc_results['predictions'] == mc_results['labels']
axes[0, 0].hist(mc_results['entropy'][correct], bins=20, alpha=0.6, label='Correct',
                color='green', edgecolor='black')
axes[0, 0].hist(mc_results['entropy'][~correct], bins=20, alpha=0.6, label='Incorrect',
                color='red', edgecolor='black')
axes[0, 0].set_xlabel('Predictive Entropy')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Uncertainty: Correct vs Incorrect')
axes[0, 0].legend()

# 2. Entropy by predicted class
for c in range(cfg.NUM_CLASSES):
    mask = mc_results['predictions'] == c
    if mask.any():
        axes[0, 1].hist(mc_results['entropy'][mask], bins=15, alpha=0.5,
                        label=cfg.CLASS_NAMES[c], color=colors[c], edgecolor='black')
axes[0, 1].set_xlabel('Predictive Entropy')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Uncertainty by Predicted Class')
axes[0, 1].legend()

# 3. Mutual information vs entropy scatter
sc = axes[0, 2].scatter(mc_results['entropy'], mc_results['mutual_info'],
                        c=[colors[p] for p in mc_results['predictions']],
                        alpha=0.6, s=20)
axes[0, 2].set_xlabel('Predictive Entropy (Total Uncertainty)')
axes[0, 2].set_ylabel('Mutual Information (Epistemic)')
axes[0, 2].set_title('Epistemic vs Total Uncertainty')
for c in range(cfg.NUM_CLASSES):
    axes[0, 2].scatter([], [], c=colors[c], label=cfg.CLASS_NAMES[c])
axes[0, 2].legend()

# 4. Reliability diagram (before calibration)
ece_b, bin_data_b = compute_ece(mc_results['mean_probs'], mc_results['labels'])
if bin_data_b:
    confs = [b[0] for b in bin_data_b]
    accs = [b[1] for b in bin_data_b]
    axes[1, 0].bar(confs, accs, width=0.06, alpha=0.7, label=f'ECE={ece_b:.4f}',
                   edgecolor='black')
    axes[1, 0].plot([0, 1], [0, 1], 'r--', label='Perfect calibration')
axes[1, 0].set_xlabel('Mean Predicted Confidence')
axes[1, 0].set_ylabel('Fraction of Positives')
axes[1, 0].set_title('Reliability Diagram (Before Temp Scaling)')
axes[1, 0].legend(); axes[1, 0].set_xlim(0, 1); axes[1, 0].set_ylim(0, 1)

# 5. Confusion matrix
cm = confusion_matrix(mc_results['labels'], mc_results['predictions'])
im = axes[1, 1].imshow(cm, interpolation='nearest', cmap='Blues')
axes[1, 1].set_title('Confusion Matrix (MC Dropout)')
axes[1, 1].set_xticks(range(cfg.NUM_CLASSES))
axes[1, 1].set_yticks(range(cfg.NUM_CLASSES))
axes[1, 1].set_xticklabels(cfg.CLASS_NAMES)
axes[1, 1].set_yticklabels(cfg.CLASS_NAMES)
axes[1, 1].set_ylabel('True'); axes[1, 1].set_xlabel('Predicted')
for i in range(cfg.NUM_CLASSES):
    for j in range(cfg.NUM_CLASSES):
        axes[1, 1].text(j, i, str(cm[i, j]), ha='center', va='center',
                        color='white' if cm[i, j] > cm.max()/2 else 'black',
                        fontsize=14, fontweight='bold')
plt.colorbar(im, ax=axes[1, 1])

# 6. ROC curves
for c in range(cfg.NUM_CLASSES):
    binary_labels = (mc_results['labels'] == c).astype(int)
    if binary_labels.sum() > 0:
        fpr, tpr, _ = roc_curve(binary_labels, mc_results['mean_probs'][:, c])
        roc_auc_val = auc(fpr, tpr)
        axes[1, 2].plot(fpr, tpr, label=f'{cfg.CLASS_NAMES[c]} (AUC={roc_auc_val:.3f})',
                        color=colors[c], linewidth=2)
axes[1, 2].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1, 2].set_xlabel('False Positive Rate')
axes[1, 2].set_ylabel('True Positive Rate')
axes[1, 2].set_title('ROC Curves (One-vs-Rest)')
axes[1, 2].legend()

plt.tight_layout()
plt.savefig(os.path.join(cfg.SAVE_DIR, 'uncertainty_analysis.png'), dpi=150)
plt.show()

# Classification report
print('\nDetailed Classification Report (MC Dropout):')
print(classification_report(
    mc_results['labels'], mc_results['predictions'],
    target_names=cfg.CLASS_NAMES, digits=4))


## 5. Explainability: 3D Grad-CAM

**Gradient-weighted Class Activation Mapping** (Selvaraju et al., 2017)
adapted for 3D volumetric inputs.

For each prediction:
1. Hook into the last conv layer (layer4) of each branch
2. Compute gradients of the target class score w.r.t. feature maps
3. Global average pool the gradients to get per-channel importance weights
4. Weighted combination of feature maps produces a 3D heatmap
5. Upsample heatmap to input volume size
6. Overlay on anatomical MRI and metabolic PET for interpretation


In [ ]:
# ============================================================
# 3D GRAD-CAM
# ============================================================

class GradCAM3D:
    """3D Grad-CAM for volumetric feature extractors.
    Hooks into the last convolutional layer and computes
    class-discriminative heatmaps.
    """

    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        # Register hooks
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, mri, pet, target_class=None, pet_conf=None):
        """Generate Grad-CAM heatmap.
        Args:
            mri: (1, 1, D, H, W)
            pet: (1, 1, D, H, W)
            target_class: int or None (uses predicted class)
            pet_conf: (1,) pet confidence tensor, or None
        Returns:
            heatmap: (D, H, W) normalized [0, 1]
            predicted_class: int
            class_probs: (num_classes,)
        """
        self.model.eval()
        mri.requires_grad_(False)
        pet.requires_grad_(False)

        # Forward
        cls_logits, _, _ = self.model(mri, pet, pet_conf)
        probs = F.softmax(cls_logits, dim=1)
        predicted_class = cls_logits.argmax(dim=1).item()

        if target_class is None:
            target_class = predicted_class

        # Backward for target class
        self.model.zero_grad()
        score = cls_logits[0, target_class]
        score.backward(retain_graph=True)

        # Grad-CAM computation
        gradients = self.gradients[0]    # (C, D', H', W')
        activations = self.activations[0]  # (C, D', H', W')

        # Global average pooling of gradients -> channel weights
        weights = gradients.mean(dim=(1, 2, 3))  # (C,)

        # Weighted combination
        heatmap = torch.zeros(activations.shape[1:], device=activations.device)
        for i, w in enumerate(weights):
            heatmap += w * activations[i]

        # ReLU (only positive contributions)
        heatmap = F.relu(heatmap)

        # Upsample to input volume size
        heatmap = heatmap.unsqueeze(0).unsqueeze(0)  # (1, 1, D', H', W')
        heatmap = F.interpolate(
            heatmap, size=mri.shape[2:], mode='trilinear', align_corners=False)
        heatmap = heatmap.squeeze().cpu().numpy()

        # Normalize to [0, 1]
        heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)

        return heatmap, predicted_class, probs[0].detach().cpu().numpy()


def visualize_gradcam_slices(mri_vol, heatmap_mri, pred_class, true_class,
                              probs, class_names, pet_vol=None, heatmap_pet=None,
                              save_path=None):
    """Visualize Grad-CAM overlaid on axial, coronal, sagittal mid-slices.

    Shows MRI + MRI Grad-CAM always.
    Only shows PET + PET Grad-CAM when pet_vol and heatmap_pet are provided
    (i.e., when real PET is available — synthetic PET is excluded).
    """
    D, H, W = mri_vol.shape
    has_pet = pet_vol is not None and heatmap_pet is not None
    n_cols = 4 if has_pet else 2  # MRI, MRI-overlay [, PET, PET-overlay]

    fig, axes = plt.subplots(3, n_cols, figsize=(5 * n_cols, 15))

    view_specs = [
        ('Axial',    lambda v: v[:, :, W//2]),
        ('Coronal',  lambda v: v[:, H//2, :]),
        ('Sagittal', lambda v: v[D//2, :, :]),
    ]

    for row, (view_name, slicer) in enumerate(view_specs):
        mri_sl = slicer(mri_vol)
        heat_mri_sl = slicer(heatmap_mri)

        # Column 0: MRI
        axes[row, 0].imshow(mri_sl, cmap='gray', vmin=-1, vmax=1)
        axes[row, 0].set_title(f'{view_name} - MRI')
        axes[row, 0].axis('off')

        # Column 1: MRI + Grad-CAM overlay
        axes[row, 1].imshow(mri_sl, cmap='gray', vmin=-1, vmax=1)
        axes[row, 1].imshow(heat_mri_sl, cmap='jet', alpha=0.4, vmin=0, vmax=1)
        axes[row, 1].set_title(f'{view_name} - MRI Grad-CAM')
        axes[row, 1].axis('off')

        if has_pet:
            pet_sl = slicer(pet_vol)
            heat_pet_sl = slicer(heatmap_pet)

            # Column 2: PET
            axes[row, 2].imshow(pet_sl, cmap='hot', vmin=-1, vmax=1)
            axes[row, 2].set_title(f'{view_name} - PET')
            axes[row, 2].axis('off')

            # Column 3: PET + Grad-CAM overlay
            axes[row, 3].imshow(pet_sl, cmap='hot', vmin=-1, vmax=1)
            axes[row, 3].imshow(heat_pet_sl, cmap='jet', alpha=0.4, vmin=0, vmax=1)
            axes[row, 3].set_title(f'{view_name} - PET Grad-CAM')
            axes[row, 3].axis('off')

    status = 'CORRECT' if pred_class == true_class else 'WRONG'
    prob_str = ' | '.join([f'{class_names[i]}: {probs[i]:.3f}' for i in range(len(class_names))])
    pet_info = 'MRI + Real PET' if has_pet else 'MRI only (synthetic PET excluded)'
    fig.suptitle(
        f'Grad-CAM | True: {class_names[true_class]} | '
        f'Pred: {class_names[pred_class]} ({status})\n'
        f'Probabilities: {prob_str}\n'
        f'Modalities shown: {pet_info}',
        fontsize=14, fontweight='bold')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()


print('3D Grad-CAM module defined.')

In [ ]:
# ============================================================
# RUN GRAD-CAM ON TEST SAMPLES
# ============================================================

# Create Grad-CAM for MRI branch (layer4)
gradcam_mri = GradCAM3D(model, model.mri_backbone.layer4)
# Create Grad-CAM for PET branch (layer4)
gradcam_pet = GradCAM3D(model, model.pet_backbone.layer4)

# Select samples: one from each class (correct predictions preferred)
n_examples = min(6, len(test_dataset))
example_indices = []

# Try to get 2 samples per class
for cls in range(cfg.NUM_CLASSES):
    class_indices = [i for i, s in enumerate(test_dataset.samples) if s[2] == cls]
    if class_indices:
        selected = random.sample(class_indices, min(2, len(class_indices)))
        example_indices.extend(selected)

print(f'Generating Grad-CAM for {len(example_indices)} test samples...\n')

for idx in example_indices:
    sample = test_dataset[idx]
    mri = sample['mri'].unsqueeze(0).to(device)
    pet = sample['pet'].unsqueeze(0).to(device)
    pet_conf = sample['pet_confidence'].unsqueeze(0).to(device)
    true_label = sample['label'].item()
    is_real_pet = sample['pet_confidence'].item() >= 1.0

    # Always generate MRI Grad-CAM (MRI is always real)
    heatmap_mri, pred_cls, probs = gradcam_mri.generate(mri, pet, pet_conf=pet_conf)

    mri_np = sample['mri'][0].numpy()

    # Only generate PET Grad-CAM and show PET slices for real PET
    if is_real_pet:
        heatmap_pet, _, _ = gradcam_pet.generate(mri, pet, pet_conf=pet_conf)
        pet_np = sample['pet'][0].numpy()
    else:
        heatmap_pet = None
        pet_np = None

    save_path = os.path.join(cfg.SAVE_DIR, f'gradcam_sample_{idx}.png')
    visualize_gradcam_slices(
        mri_np, heatmap_mri, pred_cls, true_label,
        probs, cfg.CLASS_NAMES,
        pet_vol=pet_np, heatmap_pet=heatmap_pet,
        save_path=save_path)

    pet_label = 'Real' if is_real_pet else 'Synthetic (Grad-CAM: MRI only)'
    print(f'Sample {idx}: True={cfg.CLASS_NAMES[true_label]}, '
          f'Pred={cfg.CLASS_NAMES[pred_cls]}, '
          f'PET={pet_label}, '
          f'Probs=[{", ".join([f"{p:.3f}" for p in probs])}]')

print('\nGrad-CAM analysis complete.')

## 5b. Additional Explainability (XAI) Methods

Beyond Grad-CAM, we apply multiple complementary XAI methods for robust interpretability:

| Method | Type | Reference | Key Insight |
|---|---|---|---|
| **Grad-CAM** | Gradient-based (coarse) | Selvaraju et al., 2017 | Which spatial regions are class-discriminative |
| **Integrated Gradients** | Gradient-based (fine) | Sundararajan et al., 2017 | Per-voxel attribution with axiomatic guarantees |
| **Occlusion Sensitivity** | Perturbation-based | Zeiler & Fergus, 2014 | Which regions, when removed, change the prediction |
| **Cross-Attention Maps** | Architecture-specific | — | How MRI and PET modalities attend to each other |

### Integrated Gradients
Accumulates gradients along a straight path from a baseline (zero volume) to the actual input. Satisfies **completeness** (attributions sum to the output difference) and **sensitivity** (non-zero attribution for any feature that changes the output).

### Occlusion Sensitivity
Systematically occludes 3D patches (16x16x16 voxels) and measures the drop in target class probability. Model-agnostic and intuitive — highlights regions whose removal most affects the prediction.

### Cross-Attention Maps
Extracted directly from the asymmetric cross-attention fusion module. Shows:
- **Attention Received**: which spatial tokens are most attended to (key importance)
- **Attention Given**: which spatial tokens attend most actively (query activity)
- Reveals inter-modality feature alignment between structural MRI and metabolic PET

In [ ]:
# ============================================================
# INTEGRATED GRADIENTS (Sundararajan et al., 2017)
# ============================================================

class IntegratedGradients3D:
    """Integrated Gradients for 3D volumetric models.
    Computes per-voxel attribution by accumulating gradients along
    a straight-line path from a baseline (zero volume) to the input.
    Satisfies the axioms of Completeness and Sensitivity.
    """

    def __init__(self, model):
        self.model = model

    def generate(self, mri, pet, target_class=None, pet_conf=None, n_steps=50):
        """Compute Integrated Gradients attributions for MRI and PET.
        Args:
            mri, pet:       (1, 1, D, H, W) input volumes
            target_class:   class to explain (None = predicted class)
            pet_conf:       (1,) PET confidence tensor
            n_steps:        number of interpolation steps
        Returns:
            ig_mri:  (D, H, W) MRI attribution map [0, 1]
            ig_pet:  (D, H, W) PET attribution map [0, 1]
            predicted_class: int
            class_probs:     (num_classes,)
        """
        self.model.eval()

        # Baselines: zero volumes (represents "absent information")
        baseline_mri = torch.zeros_like(mri)
        baseline_pet = torch.zeros_like(pet)

        # Get prediction
        with torch.no_grad():
            cls_logits, _, _ = self.model(mri, pet, pet_conf)
            probs = F.softmax(cls_logits, dim=1)
            predicted_class = cls_logits.argmax(dim=1).item()

        if target_class is None:
            target_class = predicted_class

        # Accumulate gradients along interpolation path
        mri_grads_sum = torch.zeros_like(mri)
        pet_grads_sum = torch.zeros_like(pet)

        for step in range(n_steps + 1):
            alpha = step / n_steps

            interp_mri = (baseline_mri + alpha * (mri - baseline_mri)).clone()
            interp_pet = (baseline_pet + alpha * (pet - baseline_pet)).clone()
            interp_mri.requires_grad_(True)
            interp_pet.requires_grad_(True)

            cls_logits, _, _ = self.model(interp_mri, interp_pet, pet_conf)
            score = cls_logits[0, target_class]

            self.model.zero_grad()
            score.backward(retain_graph=False)

            mri_grads_sum += interp_mri.grad.detach()
            pet_grads_sum += interp_pet.grad.detach()

        # IG = (input - baseline) * mean_gradient
        ig_mri = (mri - baseline_mri).detach() * (mri_grads_sum / (n_steps + 1))
        ig_pet = (pet - baseline_pet).detach() * (pet_grads_sum / (n_steps + 1))

        # Absolute value, squeeze to 3D
        ig_mri = ig_mri.squeeze().abs().cpu().numpy()
        ig_pet = ig_pet.squeeze().abs().cpu().numpy()

        # Normalize to [0, 1]
        ig_mri = (ig_mri - ig_mri.min()) / (ig_mri.max() - ig_mri.min() + 1e-8)
        ig_pet = (ig_pet - ig_pet.min()) / (ig_pet.max() - ig_pet.min() + 1e-8)

        return ig_mri, ig_pet, predicted_class, probs[0].detach().cpu().numpy()


# ---- Run Integrated Gradients on test samples ----
ig = IntegratedGradients3D(model)
print('Generating Integrated Gradients (n_steps=50) for test samples...\n')

for idx in example_indices[:3]:
    sample = test_dataset[idx]
    mri = sample['mri'].unsqueeze(0).to(device)
    pet = sample['pet'].unsqueeze(0).to(device)
    pet_conf = sample['pet_confidence'].unsqueeze(0).to(device)
    true_label = sample['label'].item()
    is_real_pet = sample['pet_confidence'].item() >= 1.0

    ig_mri, ig_pet, pred_cls, probs = ig.generate(
        mri, pet, pet_conf=pet_conf, n_steps=50)

    mri_np = sample['mri'][0].numpy()
    D, H, W = mri_np.shape

    n_cols = 4 if is_real_pet else 2
    fig, axes = plt.subplots(3, n_cols, figsize=(5 * n_cols, 15))

    views = [
        ('Axial',    lambda v: v[:, :, W // 2]),
        ('Coronal',  lambda v: v[:, H // 2, :]),
        ('Sagittal', lambda v: v[D // 2, :, :]),
    ]

    for row, (name, slicer) in enumerate(views):
        axes[row, 0].imshow(slicer(mri_np), cmap='gray', vmin=-1, vmax=1)
        axes[row, 0].set_title(f'{name} - MRI')
        axes[row, 0].axis('off')

        axes[row, 1].imshow(slicer(mri_np), cmap='gray', vmin=-1, vmax=1)
        axes[row, 1].imshow(slicer(ig_mri), cmap='hot', alpha=0.5, vmin=0, vmax=1)
        axes[row, 1].set_title(f'{name} - MRI IG Attribution')
        axes[row, 1].axis('off')

        if is_real_pet:
            pet_np = sample['pet'][0].numpy()
            axes[row, 2].imshow(slicer(pet_np), cmap='hot', vmin=-1, vmax=1)
            axes[row, 2].set_title(f'{name} - PET')
            axes[row, 2].axis('off')

            axes[row, 3].imshow(slicer(pet_np), cmap='hot', vmin=-1, vmax=1)
            axes[row, 3].imshow(slicer(ig_pet), cmap='jet', alpha=0.5, vmin=0, vmax=1)
            axes[row, 3].set_title(f'{name} - PET IG Attribution')
            axes[row, 3].axis('off')

    status = 'CORRECT' if pred_cls == true_label else 'WRONG'
    prob_str = ' | '.join(
        [f'{cfg.CLASS_NAMES[i]}: {probs[i]:.3f}' for i in range(cfg.NUM_CLASSES)])
    pet_label = 'MRI + Real PET' if is_real_pet else 'MRI only (synth PET excluded)'
    fig.suptitle(
        f'Integrated Gradients | True: {cfg.CLASS_NAMES[true_label]} | '
        f'Pred: {cfg.CLASS_NAMES[pred_cls]} ({status})\n'
        f'Probabilities: {prob_str}\nModalities shown: {pet_label}',
        fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.SAVE_DIR, f'ig_sample_{idx}.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Sample {idx}: True={cfg.CLASS_NAMES[true_label]}, '
          f'Pred={cfg.CLASS_NAMES[pred_cls]}, PET={"Real" if is_real_pet else "Synthetic"}, '
          f'Probs=[{", ".join([f"{p:.3f}" for p in probs])}]')

print('\nIntegrated Gradients analysis complete.')
gc.collect()
if device.type == 'cuda':
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# OCCLUSION SENSITIVITY ANALYSIS
# ============================================================

class OcclusionSensitivity3D:
    """3D Occlusion Sensitivity analysis (Zeiler & Fergus, 2014).
    Slides a cubic patch across the MRI volume, replaces it with
    a constant value, and measures the drop in target class probability.
    Regions whose occlusion causes the largest probability drop are
    the most important for the prediction.
    """

    def __init__(self, model):
        self.model = model

    @torch.no_grad()
    def generate(self, mri, pet, target_class=None, pet_conf=None,
                 patch_size=16, stride=16, occlude_value=0.0):
        """Generate occlusion sensitivity map for MRI modality.
        Args:
            mri, pet:       (1, 1, D, H, W) input volumes
            target_class:   class to analyze (None = predicted class)
            pet_conf:       (1,) PET confidence
            patch_size:     cubic patch side length
            stride:         step between patches
            occlude_value:  value to fill occluded region
        Returns:
            sensitivity_map: (D, H, W) normalized [0, 1]
            predicted_class: int
            class_probs:     (num_classes,)
        """
        self.model.eval()

        # Baseline prediction
        with autocast('cuda', enabled=(device.type == 'cuda')):
            cls_logits, _, _ = self.model(mri, pet, pet_conf)
        probs = F.softmax(cls_logits, dim=1)
        predicted_class = cls_logits.argmax(dim=1).item()

        if target_class is None:
            target_class = predicted_class

        baseline_prob = probs[0, target_class].item()

        D, H, W = mri.shape[2:]
        sensitivity_map = np.zeros((D, H, W), dtype=np.float32)
        count_map = np.zeros((D, H, W), dtype=np.float32)

        d_steps = list(range(0, D - patch_size + 1, stride))
        h_steps = list(range(0, H - patch_size + 1, stride))
        w_steps = list(range(0, W - patch_size + 1, stride))
        total = len(d_steps) * len(h_steps) * len(w_steps)

        pbar = tqdm(total=total, desc='Occlusion', leave=False)
        for d in d_steps:
            for h in h_steps:
                for w in w_steps:
                    mri_occ = mri.clone()
                    mri_occ[:, :,
                            d:d + patch_size,
                            h:h + patch_size,
                            w:w + patch_size] = occlude_value

                    with autocast('cuda', enabled=(device.type == 'cuda')):
                        logits_occ, _, _ = self.model(mri_occ, pet, pet_conf)
                    probs_occ = F.softmax(logits_occ, dim=1)
                    drop = max(0.0, baseline_prob - probs_occ[0, target_class].item())

                    sensitivity_map[d:d + patch_size,
                                    h:h + patch_size,
                                    w:w + patch_size] += drop
                    count_map[d:d + patch_size,
                              h:h + patch_size,
                              w:w + patch_size] += 1.0
                    pbar.update(1)
        pbar.close()

        count_map = np.maximum(count_map, 1.0)
        sensitivity_map /= count_map

        # Normalize to [0, 1]
        sensitivity_map = (sensitivity_map - sensitivity_map.min()) / \
                          (sensitivity_map.max() - sensitivity_map.min() + 1e-8)

        return sensitivity_map, predicted_class, probs[0].cpu().numpy()


# ---- Run Occlusion Sensitivity ----
occ = OcclusionSensitivity3D(model)
print('Generating Occlusion Sensitivity maps (patch=16, stride=16)...\n')

for idx in example_indices[:3]:
    sample = test_dataset[idx]
    mri = sample['mri'].unsqueeze(0).to(device)
    pet = sample['pet'].unsqueeze(0).to(device)
    pet_conf = sample['pet_confidence'].unsqueeze(0).to(device)
    true_label = sample['label'].item()

    occ_map, pred_cls, probs = occ.generate(
        mri, pet, pet_conf=pet_conf, patch_size=16, stride=16)

    mri_np = sample['mri'][0].numpy()
    D, H, W = mri_np.shape

    fig, axes = plt.subplots(3, 2, figsize=(10, 15))

    views = [
        ('Axial',    lambda v: v[:, :, W // 2]),
        ('Coronal',  lambda v: v[:, H // 2, :]),
        ('Sagittal', lambda v: v[D // 2, :, :]),
    ]

    for row, (name, slicer) in enumerate(views):
        axes[row, 0].imshow(slicer(mri_np), cmap='gray', vmin=-1, vmax=1)
        axes[row, 0].set_title(f'{name} - MRI')
        axes[row, 0].axis('off')

        axes[row, 1].imshow(slicer(mri_np), cmap='gray', vmin=-1, vmax=1)
        axes[row, 1].imshow(slicer(occ_map), cmap='jet', alpha=0.5, vmin=0, vmax=1)
        axes[row, 1].set_title(f'{name} - Occlusion Sensitivity')
        axes[row, 1].axis('off')

    status = 'CORRECT' if pred_cls == true_label else 'WRONG'
    prob_str = ' | '.join(
        [f'{cfg.CLASS_NAMES[i]}: {probs[i]:.3f}' for i in range(cfg.NUM_CLASSES)])
    fig.suptitle(
        f'Occlusion Sensitivity | True: {cfg.CLASS_NAMES[true_label]} | '
        f'Pred: {cfg.CLASS_NAMES[pred_cls]} ({status})\n'
        f'Probabilities: {prob_str}',
        fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.SAVE_DIR, f'occlusion_sample_{idx}.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Sample {idx}: True={cfg.CLASS_NAMES[true_label]}, '
          f'Pred={cfg.CLASS_NAMES[pred_cls]}, '
          f'Probs=[{", ".join([f"{p:.3f}" for p in probs])}]')

print('\nOcclusion Sensitivity analysis complete.')
gc.collect()
if device.type == 'cuda':
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# CROSS-ATTENTION MAP VISUALIZATION
# ============================================================

def visualize_attention_maps(model, dataset, indices, cfg, device):
    """Visualize cross-attention maps from the fusion module.
    Shows how MRI spatial locations attend to PET spatial locations,
    revealing which brain regions the model considers important for
    inter-modality alignment.
    """
    model.eval()

    for idx in indices:
        sample = dataset[idx]
        mri = sample['mri'].unsqueeze(0).to(device)
        pet = sample['pet'].unsqueeze(0).to(device)
        pet_conf = sample['pet_confidence'].unsqueeze(0).to(device)
        true_label = sample['label'].item()

        with torch.no_grad():
            mri_pooled, mri_spatial = model.mri_backbone(mri)
            pet_pooled, pet_spatial = model.pet_backbone(pet)
            fused, attn_weights = model.fusion(mri_spatial, pet_spatial, pet_conf)

            cls_logits = model.cls_head(fused)
            probs = F.softmax(cls_logits, dim=1)
            pred_cls = cls_logits.argmax(dim=1).item()

        # attn_weights: (B, N_target, N_source) where N = D'*H'*W'
        spatial_shape = mri_spatial.shape[2:]  # (D', H', W')
        attn = attn_weights[0].cpu().numpy()   # (N, N)

        # Average attention received by each source token
        attn_received = attn.mean(axis=0)  # (N,) — how much each token is attended to
        attn_map = attn_received.reshape(spatial_shape)

        # Average attention given by each query token
        attn_given = attn.mean(axis=1)  # (N,) — how much each token attends to others
        attn_query_map = attn_given.reshape(spatial_shape)

        # Upsample both maps to input volume size
        zoom_factors = [s / a for s, a in zip(cfg.CROP_SHAPE, spatial_shape)]
        attn_received_up = scipy_zoom(attn_map, zoom_factors, order=1)
        attn_received_up = (attn_received_up - attn_received_up.min()) / \
                           (attn_received_up.max() - attn_received_up.min() + 1e-8)

        attn_given_up = scipy_zoom(attn_query_map, zoom_factors, order=1)
        attn_given_up = (attn_given_up - attn_given_up.min()) / \
                        (attn_given_up.max() - attn_given_up.min() + 1e-8)

        mri_np = sample['mri'][0].numpy()
        D, H, W = mri_np.shape

        fig, axes = plt.subplots(3, 4, figsize=(20, 15))

        views = [
            ('Axial',    lambda v: v[:, :, W // 2]),
            ('Coronal',  lambda v: v[:, H // 2, :]),
            ('Sagittal', lambda v: v[D // 2, :, :]),
        ]

        for row, (name, slicer) in enumerate(views):
            # MRI
            axes[row, 0].imshow(slicer(mri_np), cmap='gray', vmin=-1, vmax=1)
            axes[row, 0].set_title(f'{name} - MRI')
            axes[row, 0].axis('off')

            # Attention received (key importance)
            axes[row, 1].imshow(slicer(attn_received_up), cmap='viridis',
                                vmin=0, vmax=1)
            axes[row, 1].set_title(f'{name} - Attn Received')
            axes[row, 1].axis('off')

            # Attention given (query activity)
            axes[row, 2].imshow(slicer(attn_given_up), cmap='plasma',
                                vmin=0, vmax=1)
            axes[row, 2].set_title(f'{name} - Attn Given')
            axes[row, 2].axis('off')

            # MRI + Attention overlay
            axes[row, 3].imshow(slicer(mri_np), cmap='gray', vmin=-1, vmax=1)
            axes[row, 3].imshow(slicer(attn_received_up), cmap='magma',
                                alpha=0.5, vmin=0, vmax=1)
            axes[row, 3].set_title(f'{name} - MRI + Attn Overlay')
            axes[row, 3].axis('off')

        status = 'CORRECT' if pred_cls == true_label else 'WRONG'
        prob_str = ' | '.join(
            [f'{cfg.CLASS_NAMES[i]}: {probs[0][i].item():.3f}'
             for i in range(cfg.NUM_CLASSES)])
        pet_type = 'Real' if sample['pet_confidence'].item() >= 1.0 else 'Synthetic'
        fig.suptitle(
            f'Cross-Attention Maps | True: {cfg.CLASS_NAMES[true_label]} | '
            f'Pred: {cfg.CLASS_NAMES[pred_cls]} ({status})\n'
            f'Probabilities: {prob_str} | PET: {pet_type}\n'
            f'Spatial resolution: {spatial_shape}',
            fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(os.path.join(cfg.SAVE_DIR, f'attention_sample_{idx}.png'),
                    dpi=150, bbox_inches='tight')
        plt.show()

        print(f'Sample {idx}: True={cfg.CLASS_NAMES[true_label]}, '
              f'Pred={cfg.CLASS_NAMES[pred_cls]}, Spatial={spatial_shape}')


print('Generating Cross-Attention Maps...\n')
visualize_attention_maps(model, test_dataset, example_indices[:3], cfg, device)
print('\nCross-Attention visualization complete.')

In [ ]:
# ============================================================
# COMPREHENSIVE FINAL EVALUATION
# ============================================================

print('=' * 70)
print('FINAL TEST SET EVALUATION')
print('=' * 70)

# Standard evaluation (no MC Dropout)
cls_weights_eval = compute_class_weights(train_dataset, cfg.NUM_CLASSES, device)
cls_criterion = nn.CrossEntropyLoss(weight=cls_weights_eval)
standard_results = evaluate(model, test_loader, cls_criterion, cfg)

print(f'\nStandard Evaluation:')
print(f'  Accuracy:          {standard_results["accuracy"]:.4f}')
print(f'  Balanced Accuracy: {standard_results["balanced_accuracy"]:.4f}')
print(f'  MMSE MAE:          {standard_results["mmse_mae"]:.2f} points')

print(f'\nMC Dropout Evaluation ({cfg.MC_SAMPLES} samples):')
print(f'  Accuracy:          {mc_acc:.4f}')
print(f'  Balanced Accuracy: {mc_bal_acc:.4f}')

# Per-class sensitivity and specificity
cm = confusion_matrix(mc_results['labels'], mc_results['predictions'])
print(f'\nPer-Class Metrics:')
print(f'{"Class":>8s} {"SEN":>8s} {"SPE":>8s} {"F1":>8s} {"N":>6s}')
for c in range(cfg.NUM_CLASSES):
    tp = cm[c, c]
    fn = cm[c, :].sum() - tp
    fp = cm[:, c].sum() - tp
    tn = cm.sum() - tp - fn - fp
    sen = tp / max(1, tp + fn)
    spe = tn / max(1, tn + fp)
    f1 = 2 * tp / max(1, 2 * tp + fp + fn)
    print(f'{cfg.CLASS_NAMES[c]:>8s} {sen:8.4f} {spe:8.4f} {f1:8.4f} {tp+fn:6d}')

# Multi-class AUC
try:
    auc_macro = roc_auc_score(
        mc_results['labels'], mc_results['mean_probs'],
        multi_class='ovr', average='macro')
    print(f'\nMacro AUC (OVR): {auc_macro:.4f}')
except ValueError as e:
    print(f'\nAUC computation error: {e}')

# ECE
ece_test, _ = compute_ece(mc_results['mean_probs'], mc_results['labels'])
print(f'\nCalibration:')
print(f'  ECE (before temp scaling):  {ece_test:.4f}')
print(f'  ECE (after temp scaling):   {ece_after:.4f}')
print(f'  Temperature:                {temp_model.temperature.item():.4f}')

# MMSE regression results
if mc_results['mmse_valid'].any():
    pred_mmse = mc_results['mmse_mean'][mc_results['mmse_valid']] * 30.0
    true_mmse = mc_results['mmse_true'][mc_results['mmse_valid']] * 30.0
    mmse_mae = mean_absolute_error(true_mmse, pred_mmse)
    mmse_rmse = np.sqrt(mean_squared_error(true_mmse, pred_mmse))
    print(f'\nMMSE Regression:')
    print(f'  MAE:  {mmse_mae:.2f} points')
    print(f'  RMSE: {mmse_rmse:.2f} points')

print(f'\n{"=" * 70}')

# Save results
results_dict = {
    'accuracy': float(mc_acc),
    'balanced_accuracy': float(mc_bal_acc),
    'ece_before': float(ece_test),
    'ece_after': float(ece_after),
    'temperature': float(temp_model.temperature.item()),
    'per_class_confusion': cm.tolist(),
}
with open(os.path.join(cfg.SAVE_DIR, 'test_results.json'), 'w') as f:
    json.dump(results_dict, f, indent=2)
print(f'Results saved to {cfg.SAVE_DIR}/test_results.json')


## Summary

### Architecture

| Component | Detail |
|---|---|
| **MRI Branch** | 3D ResNet-18 (1ch input, 512-dim features) |
| **PET Branch** | 3D ResNet-18 (1ch input, 512-dim features) |
| **Fusion** | 2-layer asymmetric cross-attention (8 heads) with confidence-weighted gating |
| **Classification** | 3-class (CN/MCI/AD), cross-entropy with class weights |
| **Regression** | MMSE score prediction (Smooth L1 loss) |
| **Training** | AdamW, cosine LR with warmup, mixed precision FP16 |

### Real vs Synthetic PET Handling

| PET Source | Confidence | Effective Fusion Weight |
|---|---|---|
| **Real PET** | 1.0 | Gate learns freely (balanced MRI/PET) |
| **Synthetic PET** | 0.3 | MRI ~70%, PET ~30% (gate biased toward MRI) |

The confidence weighting operates at two levels:
1. **Feature scaling**: PET spatial tokens are multiplied by confidence before cross-attention
2. **Gate biasing**: `final_gate = raw_gate * pet_conf + (1 - pet_conf)`, ensuring MRI dominance for synthetic PET

### Uncertainty & Explainability

| Method | Type | Purpose |
|---|---|---|
| **MC Dropout** | Uncertainty | Epistemic uncertainty via 30 stochastic forward passes |
| **Temperature Scaling** | Calibration | Post-hoc calibration, measured by ECE |
| **3D Grad-CAM** | XAI (gradient, coarse) | Class-discriminative heatmaps on MRI/PET volumes |
| **Integrated Gradients** | XAI (gradient, fine) | Per-voxel attribution with completeness and sensitivity axioms |
| **Occlusion Sensitivity** | XAI (perturbation) | Region importance via systematic 3D patch occlusion |
| **Cross-Attention Maps** | XAI (architecture) | Inter-modality attention patterns (received and given) |

### Metrics
- **Classification**: Accuracy, Balanced Accuracy, Sensitivity, Specificity, F1, AUC
- **Regression**: MMSE MAE, RMSE
- **Calibration**: ECE before/after temperature scaling
- **Uncertainty**: Predictive entropy, mutual information

### Outputs
- `best_model.pt` - Best checkpoint by balanced accuracy
- `latest_checkpoint.pt` - Resume checkpoint
- `training_curves.png` - Loss and accuracy plots
- `data_distribution.png` - Class and MMSE distributions
- `uncertainty_analysis.png` - Entropy, calibration, ROC, confusion matrix
- `gradcam_sample_*.png` - Grad-CAM visualizations per sample
- `ig_sample_*.png` - Integrated Gradients attributions per sample
- `occlusion_sample_*.png` - Occlusion sensitivity maps per sample
- `attention_sample_*.png` - Cross-attention maps per sample
- `xai_comparison_sample_*.png` - Side-by-side comparison of all XAI methods
- `test_results.json` - Final numeric results